# My first assignment
## LLM Assignment: Website Summarization with LLaMA 3.2

In this assignment, I built a simple pipeline that:
- Scrapes text content from a given website URL  
- Uses **LLaMA 3.2** as a Large Language Model (LLM) to generate a concise summary of the page  
- Displays the final summarized text  

The goal of this task is to practice combining **web scraping** with **LLM-based summarization** in a Jupyter Notebook.


In [11]:
import requests
import ollama


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from IPython.display import Markdown, display

MODEL = "llama3.2"

### Website Scraping Class
Create a class to fetch the page title and clean text from a given URL.


In [2]:
class Website:
    def __init__(self, url):
        self.url = url

        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                             "AppleWebKit/537.36 (KHTML, like Gecko) "
                             "Chrome/117.0.0.0 Safari/537.36")

        # 👇 add these 3 lines (anti-detection)
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        options.add_argument("--disable-blink-features=AutomationControlled")

        # Start Selenium
        service = ChromeService(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)

        try:
            driver.get(url)
            html = driver.page_source   # rendered HTML
            soup = bs(html, "html.parser")

            # Title
            self.title = soup.title.string if soup.title else "no title found"

            # Clean irrelevant tags
            for irrelevant in soup(["script", "style", "img", "input"]):
                irrelevant.decompose()

            # Extract text
            body = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
            self.text = body

        finally:
            driver.quit()


In [3]:
ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Skip to content
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press 

### System Prompt
Define the instructions that guide the LLM on how to summarize the website.


In [4]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

### User Prompt Function
Builds the user prompt by combining the website title and text to send to the LLM.


In [5]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

### Message Builder
Combine the system and user prompts into a message list for the LLM.


In [6]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

### Summarization Function
Scrape the website and generate a summary using the LLM.


In [7]:
def summarize(url):
    website = Website(url)
    response = ollama.chat(model=MODEL,messages=messages_for(website))
    return response['message']['content']

In [8]:
summarize("https://edwarddonner.com")

'**Website Summary**\n======================\n\n### Overview\nThis website belongs to Edward Donner, a co-founder and CTO of Nebula.io. The site showcases his personal projects, expertise, and accomplishments in the field of Artificial Intelligence (AI).\n\n### Recent News and Announcements\n---------------------------\n\n*   **2025 AI Executive Briefing** (April 21, 2025)\n*   **The Complete Agentic AI Engineering Course** (May 18, 2025)\n*   **Connecting my courses – become an LLM expert and leader** (May 28, 2025)\n*   **AI in Production: Gen AI and Agentic AI on AWS at scale** (September 15, 2025)\n\n### About the Author\n--------------------\n\nEdward Donner is a writer, DJ, amateur electronic music producer, and entrepreneur. He founded and sold his previous AI startup, untapt, acquired in 2021.\n\n### Contact Information\n----------------------\n\n*   Email: `ed [at] edwarddonner [dot] com`\n*   Website: `www.edwarddonner.com`\n*   Social Media:\n    *   LinkedIn\n    *   Twitte

### Display Function
Show the website summary in a nicely formatted markdown output.


In [12]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [13]:
display_summary("https://edwarddonner.com")

**Summary**
=================

The website "Home - Edward Donner" appears to be a personal blog or portfolio site for Edward Donner, the co-founder and CTO of Nebula.io. The site showcases his interests in artificial intelligence (AI), writing, music production, and DJing.

**News and Announcements**
---------------------------

* **Upcoming Events:**
	+ September 15, 2025: "AI in Production: Gen AI and Agentic AI on AWS at scale" 
	+ May 28, 2025: "Connecting my courses – become an LLM expert and leader"
	+ May 18, 2025: "2025 AI Executive Briefing"
	+ April 21, 2025: "The Complete Agentic AI Engineering Course"

**Other Notes**
---------------

* The website also features a section called "Connect Four," which appears to be an arena where large language models (LLMs) compete against each other in a battle of diplomacy and deviousness.
* There is a contact form and links to social media profiles, including LinkedIn, Twitter, Facebook, and newsletter subscription.

In [15]:
display_summary("https://anthropic.com")

# Anthropic Website Summary

## Mission and Purpose
Anthropic is a public benefit corporation dedicated to securing the benefits of AI while mitigating its risks. The company aims to build tools with human benefit at their foundation, like Claude.

## Research and Initiatives
Anthropic conducts research on AI's effects on the labor market and broader economy over time through the Anthropic Economic Index. The company also focuses on building responsible AI development practices, including its Responsible Scaling Policy.

### News and Announcements

*   **Claude Opus 4.1**: Released on August 12, 2025.
*   **Claude Sonnet 4 with 1M context**: Published on September 15, 2025.
*   **Project Vend**: Announced on August 5, 2025.
*   **Introducing Claude 4**: Announced on May 22, 2025.
*   **Tracing the thoughts of a large language model**: Discussed on March 27, 2025.

## Products
Anthropic offers several products and services, including:

### AI agents

*   **Claude Code**
*   **Claude Developer Platform**

### Solutions

*   **AI agents**
*   **Code modernization**
*   **Coding**
*   **Customer support**
*   **Education**
*   **Financial services**
*   **Government**

## Partnerships
Anthropic partners with various organizations, including:

*   **Amazon Bedrock**
*   **Google Cloud’s Vertex AI**

In [16]:
display_summary("https://cnn.com")

**Summary of CNN Website**
==========================

The CNN website provides breaking news, in-depth analysis, and feature articles on various topics such as:

### News

* **Ukraine-Russia War**: Ukraine's air defenses intercept a Russian attack over Kyiv.
* **Israel-Hamas War**: International condemnation soars over Israel's conduct in Gaza.
* **Trump Plans to Attend Military Gathering**: President Trump plans to attend a gathering of senior military officials in Virginia.
* **Iran Sanctions**: Iran hit with "snapback" sanctions over its nuclear program.

### Politics

* **US Politics**: Top congressional leaders will meet Trump at the White House on Monday as shutdown looms.
* **Trump Accuses FBI Director Wray of Lying**: President Trump accuses FBI Director Christopher Wray of lying about agency actions on January 6.
* **James Comey Indictment**: The case against former FBI Director James Comey is different from the Biden DOJ's indictment of Trump.

### Business

* **China's Electric Car Market Boom Turns Bloodbath**: China's electric car market has become a bloodbath as boom turns into bust.

### Entertainment

* **Selena Gomez Weds Benny Blanco**: Singer Selena Gomez marries music producer Benny Blanco.
* **Meryl Streep Crashes Milan Fashion Show for 'Devil Wears Prada 2' Shoot**: Actress Meryl Streep crashes a fashion show to film a scene from the sequel.

### Science and Technology

* **New Mission to Map the Farthest Spacecraft**: A new mission will map the farthest spacecraft from Earth.
* **DNA Evidence Links Dead Man to 1991 Yogurt Shop Murders**: DNA evidence links a dead man to the 1991 killings of four girls at a Texas yogurt shop.

### Sports

* **Team Europe Wins Ryder Cup in Shellacking of Team USA**: European team wins the Ryder Cup with a dominating performance against Team USA.
* **American Wins Top Cheese Competition**: An American makes history by winning one of France's top cheese competitions.